In [1]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 55.3 MB/s eta 0:00:00:00:0100:01


# **1. Retrieval model**

In [2]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import glob
import fitz

# KHÁC BIỆT Ở ĐÂY: Thay đổi tên mô hình thành BAAI/bge-m3
# Mô hình này hỗ trợ tốt cả tiếng Anh và tiếng Việt
model = SentenceTransformer('BAAI/bge-m3')

# 1. Khởi tạo một mảng (list) rỗng để chứa tất cả các đoạn văn
documents = []

# 2. Tìm tất cả các file có đuôi .txt trong thư mục hiện tại
cac_file_txt = glob.glob("/kaggle/input/datasets/huydoanlame/rag-document-data/*.txt")
cac_file_pdf = glob.glob("/kaggle/input/datasets/huydoanlame/rag-document-data/*.pdf")

print(f"Tìm thấy {len(cac_file_txt)} file .txt và {len(cac_file_pdf)} file .pdf. Đang tiến hành đọc dữ liệu...\n")

# 3. Duyệt qua từng file để đọc nội dung
for ten_file in cac_file_txt:
    # Mở file ở chế độ 'r' (read - đọc)
    with open(ten_file, 'r', encoding='utf-8') as f:
        # Đọc tất cả các dòng trong file
        lines = f.readlines()

        for line in lines:
            text = line.strip()  # Cắt bỏ khoảng trắng và dấu xuống dòng thừa

            # Chỉ đưa vào mảng nếu dòng đó có chữ (không rỗng)
            # VÀ không phải là dòng kẻ phân cách (ví dụ: '---') mà chúng ta có thể đã tạo trước đó
            if text and not text.startswith('---'):
                documents.append(text)

for file_pdf in cac_file_pdf:
    with fitz.open(file_pdf) as pdf:
        for trang in pdf:
            cac_doan = trang.get_text().split('\n')
            for doan in cac_doan:
                chu_sach = doan.strip()
                if chu_sach:
                    documents.append(chu_sach)

# 4. Kiểm tra kết quả
print(f"Tổng số đoạn văn đã gom được vào mảng: {len(documents)}")

# In thử 3 đoạn văn đầu tiên trong mảng để kiểm tra
print("\n--- 10 phần tử đầu tiên trong mảng ---")
for i in range(min(10, len(documents))):
    print(f"Phần tử {i}: {documents[i]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Tìm thấy 12 file .txt và 5 file .pdf. Đang tiến hành đọc dữ liệu...

Tổng số đoạn văn đã gom được vào mảng: 37401

--- 10 phần tử đầu tiên trong mảng ---
Phần tử 0: Trường Đại học Công nghệ (ĐHCN), Đại học Quốc gia Hà Nội (ĐHQGHN) hiện triển khai đào tạo bậc đại học với 20 ngành đào tạo với 22 chương trình đào tạo Cử nhân, Kỹ sư.  Nhà trường tổ chức đào tạo các ngành học hiện đại đáp ứng nhu cầu phát triển nguồn nhân lực trong bối cảnh cách mạng công nghiệp lần thứ tư bao gồm các ngành thuộc các lĩnh vực công nghệ thông tin, truyền thông, tự động hóa và một số ngành công nghệ cao liên ngành.
Phần tử 1: Với chương trình đào tạo chất lượng cao, sinh viên được học bổ sung một số học phần nâng cao về chuyên môn và kỹ năng, trong đó tối thiêu 40% môn học chuyên ngành được giảng dạy bằng tiếng Anh.
Phần tử 2: Sinh viên được thực hành thực tập trên hệ thống trang thiết bị và phòng máy tính hiện đại, được cung cấp tài nguyên học liệu số. 100% môn học có website hỗ trợ giảng dạy, học tập. Từ họ

# **2. Extracting model** 

In [3]:
from transformers import pipeline

questions = ['Mục tiêu tổng quát đến năm 2030 của hoạt động KHCN&ĐMST là đưa ĐHQGHN vào nhóm bao nhiêu đại học hàng đầu thế giới?',
             'Who defeated Roger Federer in the men\'s singles final at the 2012 London Olympics?',
             '¿En qué liga juega el equipo actual de Cristiano Ronaldo?'
             ]



qa_pipeline = pipeline("question-answering", 'deepset/xlm-roberta-base-squad2')

for question in questions:
    print(question)
    # result = qa_pipeline(question=question, context=" ".join(documents))
    result = qa_pipeline(question=question, context="\n".join(documents))
    print(result)
    print("\n")


config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Mục tiêu tổng quát đến năm 2030 của hoạt động KHCN&ĐMST là đưa ĐHQGHN vào nhóm bao nhiêu đại học hàng đầu thế giới?
{'score': 1.6253127258678433, 'start': 19715, 'end': 19718, 'answer': '300'}


Who defeated Roger Federer in the men's singles final at the 2012 London Olympics?
{'score': 1.3650140481751407, 'start': 672866, 'end': 672881, 'answer': '"Novak Djokovic'}


¿En qué liga juega el equipo actual de Cristiano Ronaldo?
{'score': 1.3105015950277448, 'start': 204536, 'end': 204561, 'answer': 'Liga Profesional Saudí.19'}




# **3. Generating model**

## a. Prompting model

In [4]:
from transformers import pipeline

# 1. Khởi tạo pipeline sinh văn bản (Text Generation)
generator = pipeline("text-generation", model="Qwen/Qwen1.5-1.8B-Chat")

# 2. Giả sử đây là tài liệu bạn vừa truy xuất được từ hệ thống Retrieval
retrieved_context = "Trường Đại học Công nghệ (UET) là một trường đại học thành viên của Đại học Quốc gia Hà Nội, được thành lập vào năm 2004."
question = "UET được thành lập vào năm nào?"

# 3. Tạo Prompt (Cấu trúc đầu vào)
prompt = f"""Bạn là một trợ lý hữu ích. Hãy dựa vào ngữ cảnh sau đây để trả lời câu hỏi. 
Nếu thông tin không có trong ngữ cảnh, hãy nói 'Tôi không biết'.

Ngữ cảnh: {retrieved_context}
Câu hỏi: {question}
Trả lời ngắn gọn:"""

# 4. Chạy mô hình
output = generator(prompt, max_new_tokens=20, do_sample=False)
print(output[0]['generated_text'])

# Kết quả mong đợi: "Năm 2004."

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bạn là một trợ lý hữu ích. Hãy dựa vào ngữ cảnh sau đây để trả lời câu hỏi. 
Nếu thông tin không có trong ngữ cảnh, hãy nói 'Tôi không biết'.

Ngữ cảnh: Trường Đại học Công nghệ (UET) là một trường đại học thành viên của Đại học Quốc gia Hà Nội, được thành lập vào năm 2004.
Câu hỏi: UET được thành lập vào năm nào?
Trả lời ngắn gọn: UET được thành lập vào năm 2004.


## b. Training model

In [5]:
import json
import os

# 1. Đọc câu hỏi và câu trả lời
with open("/kaggle/input/datasets/huydoanlame/rag-qa-data/data/train/questions.txt", "r", encoding="utf-8") as f_q:
    questions = [line.strip() for line in f_q.readlines()]

with open("/kaggle/input/datasets/huydoanlame/rag-qa-data/data/train/reference_answers.txt", "r", encoding="utf-8") as f_a:
    answers = [line.strip() for line in f_a.readlines()]

# 2. ĐẢM BẢO RẰNG: list_of_paragraphs chứa CÁC ĐOẠN VĂN THẬT từ PDF/TXT của bạn
# (Nếu bạn chưa làm bước đọc file PDF vào list này, bạn phải thực hiện trước)
# list_of_paragraphs = [
#     # ... Text thật của bạn phải nằm ở đây ...
#     "Trường Đại học Công nghệ (UET) được thành lập vào năm 2004, là một trường đại học thành viên...",
#     "Carnegie Mellon University was founded in 1900 by Andrew Carnegie."
# ]

training_data = []

# 3. Ghép nối và tìm ngữ cảnh (Đã nâng cấp logic tìm kiếm)
for q, a in zip(questions, answers):
    if not q or not a: # Bỏ qua dòng trống
        continue
        
    possible_answers = a.split(";") 
    first_answer = possible_answers[0].strip()
    
    matched_context = None
    answer_start = -1
    
    for para in documents:
        # TÌM KIẾM KHÔNG PHÂN BIỆT HOA/THƯỜNG
        # Chuyển cả đoạn văn và câu trả lời về chữ thường để so sánh
        idx = para.lower().find(first_answer.lower())
        
        if idx != -1: # Nếu tìm thấy (hàm find trả về vị trí >= 0)
            matched_context = para
            answer_start = idx # Vị trí token bắt đầu
            break
            
    if matched_context:
        data_point = {
            "context": matched_context,
            "question": q,
            "answers": {
                "text": [first_answer],
                "answer_start": [answer_start]
            }
        }
        training_data.append(data_point)

# KIỂM TRA QUAN TRỌNG: In ra số lượng để xem code có tìm được ngữ cảnh không
print(f"Tổng số câu hỏi: {len(questions)}")
print(f"Số lượng ghép thành công: {len(training_data)}")

# 4. Lưu ra file JSON (Chỉ lưu nếu có dữ liệu)
if len(training_data) > 0:
    output_dir = "/kaggle/working/data/train"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, "train_data.json")
    with open(output_file, "w", encoding="utf-8") as f_out:
        json.dump(training_data, f_out, ensure_ascii=False, indent=4)
    print(f"Đã lưu thành công tại: {output_file}")
else:
    print("CẢNH BÁO: Không có câu hỏi nào khớp với tài liệu. Hãy kiểm tra lại list_of_paragraphs và reference_answers.txt!")

Tổng số câu hỏi: 270
Số lượng ghép thành công: 211
Đã lưu thành công tại: /kaggle/working/data/train/train_data.json


In [6]:
import json
import os

# 1. Đọc câu hỏi và câu trả lời
with open("/kaggle/input/datasets/huydoanlame/rag-qa-data/data/test/questions.txt", "r", encoding="utf-8") as f_q:
    questions = [line.strip() for line in f_q.readlines()]

with open("/kaggle/input/datasets/huydoanlame/rag-qa-data/data/test/reference_answers.txt", "r", encoding="utf-8") as f_a:
    answers = [line.strip() for line in f_a.readlines()]

# 2. ĐẢM BẢO RẰNG: list_of_paragraphs chứa CÁC ĐOẠN VĂN THẬT từ PDF/TXT của bạn
# (Nếu bạn chưa làm bước đọc file PDF vào list này, bạn phải thực hiện trước)
# list_of_paragraphs = [
#     # ... Text thật của bạn phải nằm ở đây ...
#     "Trường Đại học Công nghệ (UET) được thành lập vào năm 2004, là một trường đại học thành viên...",
#     "Carnegie Mellon University was founded in 1900 by Andrew Carnegie."
# ]

testing_data = []

# 3. Ghép nối và tìm ngữ cảnh (Đã nâng cấp logic tìm kiếm)
for q, a in zip(questions, answers):
    if not q or not a: # Bỏ qua dòng trống
        continue
        
    possible_answers = a.split(";") 
    first_answer = possible_answers[0].strip()
    
    matched_context = None
    answer_start = -1
    
    for para in documents:
        # TÌM KIẾM KHÔNG PHÂN BIỆT HOA/THƯỜNG
        # Chuyển cả đoạn văn và câu trả lời về chữ thường để so sánh
        idx = para.lower().find(first_answer.lower())
        
        if idx != -1: # Nếu tìm thấy (hàm find trả về vị trí >= 0)
            matched_context = para
            answer_start = idx # Vị trí token bắt đầu
            break
            
    if matched_context:
        data_point = {
            "context": matched_context,
            "question": q,
            "answers": {
                "text": [first_answer],
                "answer_start": [answer_start]
            }
        }
        testing_data.append(data_point)

# KIỂM TRA QUAN TRỌNG: In ra số lượng để xem code có tìm được ngữ cảnh không
print(f"Tổng số câu hỏi: {len(questions)}")
print(f"Số lượng ghép thành công: {len(testing_data)}")

# 4. Lưu ra file JSON (Chỉ lưu nếu có dữ liệu)
if len(testing_data) > 0:
    output_dir = "/kaggle/working/data/test"
    os.makedirs(output_dir, exist_ok=True)
    
    output_file = os.path.join(output_dir, "test_data.json")
    with open(output_file, "w", encoding="utf-8") as f_out:
        json.dump(testing_data, f_out, ensure_ascii=False, indent=4)
    print(f"Đã lưu thành công tại: {output_file}")
else:
    print("CẢNH BÁO: Không có câu hỏi nào khớp với tài liệu. Hãy kiểm tra lại list_of_paragraphs và reference_answers.txt!")

Tổng số câu hỏi: 30
Số lượng ghép thành công: 23
Đã lưu thành công tại: /kaggle/working/data/test/test_data.json


In [7]:
from datasets import load_dataset

# Tải dữ liệu từ file JSON vừa tạo
dataset = load_dataset("json", data_files={"train": "/kaggle/working/data/train/train_data.json", 
                                           "test": '/kaggle/working/data/test/test_data.json'})

print(dataset["train"][0]) 
# In ra thử 1 mẫu để kiểm tra

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

{'context': 'Đại học Quốc gia Hà Nội (ĐHQGHN) là trung tâm đào tạo, nghiên cứu khoa học, công nghệ đa ngành, đa lĩnh vực chất lượng cao, được Nhà nước ưu tiên đầu tư phát triển. Hiện tại, ĐHQGHN đang triển khai 190 chương trình đào tạo đại học và 198 chương trình đào tạo thạc sỹ và 118 chương trình đào tạo tiến sỹ thuộc lĩnh vực khoa học tự nhiên, công nghệ, khoa học xã hội nhân văn, kinh tế, giáo dục, ngoại ngữ… Tỷ lệ quy mô đào tạo sau đại học/tổng quy mô đào tạo chính quy đạt 33.88%.\xa0Quy mô đào tạo đại học chính quy được giữ ổn định. Hàng năm, ĐHQGHN đào tạo trên 5.000 cử nhân khoa học, trong đó 13.1% sinh viên tài năng, chất lượng cao, tiên tiến, đạt chuẩn quốc tế; 2.400 thạc sỹ và\xa0200 tiến sỹ.', 'question': 'Tỷ lệ quy mô đào tạo sau đại học trên tổng quy mô đào tạo chính quy của ĐHQGHN đạt bao nhiêu phần trăm?', 'answers': {'text': ['33.88%'], 'answer_start': [471]}}


In [8]:
!pip install trl
!pip install evaluate rouge_score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.8/760.8 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [9]:
import numpy as np
import evaluate

rouge_metric = evaluate.load("rouge")
exact_match_metric = evaluate.load("exact_match")

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    
    # 🌟 LẤY MÃ TOKEN AN TOÀN (Đảm bảo là số nguyên, không bị None) 🌟
    safe_pad_id = tokenizer.eos_token_id if tokenizer.pad_token_id is None else tokenizer.pad_token_id

    # Thay thế các token -100 bằng mã token an toàn
    labels = np.where(labels != -100, labels, safe_pad_id)
    preds = np.where(preds != -100, preds, safe_pad_id)
    
    # Ép kiểu tường minh về số nguyên để thư viện mã hóa không bị lỗi
    labels = labels.astype(np.int64)
    preds = preds.astype(np.int64)

    # Dịch ngược từ số (ID) sang văn bản (Text)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Xóa các khoảng trắng thừa
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # Tính toán các chỉ số NLP
    rouge_result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    em_result = exact_match_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "rouge1": round(rouge_result["rouge1"], 4),
        "rougeL": round(rouge_result["rougeL"], 4),
        "exact_match": round(em_result["exact_match"], 4)
    }

In [10]:
import os
from datasets import load_dataset
import math
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments

# --- BƯỚC 1: LÀM MỚI ĐƯỜNG DẪN ĐỂ LOAD CẢ TRAIN VÀ TEST ---
# # Giả sử bạn đã tạo file test_data.json tương tự như cách làm file train
# dataset = load_dataset("json", data_files={
#     "train": "/kaggle/working/data/train/train_data.json",
#     "test": "/kaggle/working/data/test/test_data.json"  # Đường dẫn tới file test của bạn
# })

def format_prompts_for_gpt2(example):
    # Dùng các từ khóa rõ ràng như "Ngữ cảnh:", "Câu hỏi:", "Trả lời:" 
    # và kết thúc bằng <|endoftext|>
    formatted_text = f"""Dựa vào ngữ cảnh dưới đây, hãy trả lời câu hỏi thật ngắn gọn.
    Ngữ cảnh: {example['context']}
    Câu hỏi: {example['question']}
    Trả lời: {example['answers']['text'][0]}<|endoftext|>"""
    
    return {"text": formatted_text}

# Hàm map này sẽ tự động áp dụng cho CẢ HAI tập train và test có trong dataset
dataset = dataset.map(format_prompts_for_gpt2)

# 2. Khởi tạo LLM và Tokenizer
model_id = "openai-community/gpt2"
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# tokenizer.model_max_length = 512

tokenizer.padding_side = "right"

# --- BƯỚC 2: CẬP NHẬT THAM SỐ HUẤN LUYỆN ĐỂ ĐÁNH GIÁ (EVALUATION) ---
training_args = SFTConfig(
    output_dir="./results_llm_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    save_strategy="epoch",
    eval_strategy="epoch",
    per_device_eval_batch_size=2,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    
    # 🌟 DI CHUYỂN 2 THAM SỐ GÂY LỖI VÀO ĐÂY 🌟
    dataset_text_field="text",   # Chỉ định cột dữ liệu văn bản ở đây
    max_length=1024          # Chỉ định độ dài chuỗi tối đa ở đây
)

# --- BƯỚC 3: TRUYỀN TẬP TEST VÀO SFTTRAINER ---
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args, # Trainer sẽ tự động đọc dataset_text_field từ trong training_args ra
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics
)

# 4. Bắt đầu huấn luyện và đánh giá
trainer.train()

Map:   0%|          | 0/211 [00:00<?, ? examples/s]

Map:   0%|          | 0/23 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=42, training_loss=1.7143093858446394, metrics={'train_runtime': 124.4721, 'train_samples_per_second': 5.085, 'train_steps_per_second': 0.337, 'total_flos': 213190822656000.0, 'train_loss': 1.7143093858446394, 'epoch': 3.0})

In [11]:
# 1. Định nghĩa đường dẫn thư mục muốn lưu trên Kaggle
output_model_path = "/kaggle/working/my_fine_tuned_model"

# 2. Lưu mô hình (bao gồm file trọng số weights và file config)
trainer.save_model(output_model_path)

# 3. Lưu kèm tokenizer vào cùng thư mục (bắt buộc phải lưu để sau này dịch câu hỏi)
tokenizer.save_pretrained(output_model_path)

print(f"Đã lưu mô hình và tokenizer thành công tại: {output_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu mô hình và tokenizer thành công tại: /kaggle/working/my_fine_tuned_model


In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Đường dẫn tới thư mục mô hình đã lưu ở bước trước
model_path = "/kaggle/working/my_fine_tuned_model"

# 2. Tải lại mô hình và tokenizer vào GPU (nếu có)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# 3. Chuẩn bị Ngữ cảnh và Câu hỏi test
context = "Trường Đại học Công nghệ (UET) - ĐHQGHN được thành lập năm 2004 theo Quyết định của Thủ tướng Chính phủ trên cơ sở Khoa Công nghệ thuộc ĐHQGHN."
question = "Trường UET được thành lập vào năm nào?"

# 4. Tạo Prompt định dạng CHÍNH XÁC như lúc Train (nhưng dừng lại ở thẻ mở của Model)
# Lưu ý: Nếu bạn dùng GPT-2 thì sửa lại theo format của GPT-2 nhé!
prompt = f"""<bos><start_of_turn>user
Dựa vào ngữ cảnh dưới đây, hãy trả lời câu hỏi thật ngắn gọn.
Ngữ cảnh: {context}
Câu hỏi: {question}<end_of_turn>
<start_of_turn>model
"""

# 5. Mã hóa Prompt thành các mã số (Tokens) và đưa lên GPU
inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_length = inputs.input_ids.shape[1] # Độ dài của prompt đầu vào

# 6. Yêu cầu mô hình sinh văn bản (Generate)
with torch.no_grad(): # Tắt tính toán đạo hàm để chạy nhanh và tiết kiệm VRAM
    output_ids = model.generate(
        **inputs,
        max_new_tokens=500,      # Giới hạn tối đa câu trả lời (vì đề bài yêu cầu ngắn gọn)
        do_sample=False,        # Đặt bằng False (Greedy Search) để câu trả lời chính xác và cố định
        eos_token_id=tokenizer.eos_token_id # Dừng lại ngay khi gặp token kết thúc câu
    )

# 7. CẮT BỎ PHẦN PROMPT - Chỉ lấy phần token mô hình mới sinh ra
generated_tokens = output_ids[0][input_length:]

# 8. Giải mã số thành chữ thuần túy
answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi: {question}")
print(f"Mô hình trả lời: {answer}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

--- KẾT QUẢ INFERENCE ---
Câu hỏi: Trường UET được thành lập vào năm nào?
Mô hình trả lời: Nhân: Trường Đại học Công nghệ (UET) - ĐHQGHN được thành lập năm 2004 theo Quyết định của Thủ tướng Chính phủ trên cơ sở Khoa Công nghệ thuộc ĐHQGHN.
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET được thành lập vào năm nào?<end_of_turn>
Nhân: Trường UET đ


In [13]:
import json
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# 1. Load mô hình và tập dữ liệu test gốc
model_path = "/kaggle/working/my_fine_tuned_model"

model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Đọc file test dữ liệu thô
test_dataset = load_dataset("json", data_files={"test": "/kaggle/working/data/test/test_data.json"})["test"]

results = []

print("Đang tiến hành Inference trên tập Test...")
for item in tqdm(test_dataset):
    ctx = item["context"]
    q = item["question"]
    true_answers = item["answers"]["text"] 
    
    # SỬA LẠI PROMPT THUẦN (Đồng bộ với cấu trúc lúc bạn Train GPT-2)
    # Tránh dùng <start_of_turn> của Gemma nếu chưa resize embedding lúc train
    prompt = f"""Dựa vào ngữ cảnh dưới đây, hãy trả lời câu hỏi thật ngắn gọn.
Ngữ cảnh: {ctx}
Câu hỏi: {q}
Trả lời: """
    
    # 🌟 SỬA TẠI ĐÂY: Thêm truncation và max_length để bảo vệ GPU không bị tràn chỉ mục
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        truncation=True, 
        max_length=950 # Giới hạn an toàn để GPT-2 không bị quá 1024 tokens khi cộng thêm 50 tokens generate
    ).to(model.device)
    
    input_length = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )
        
    generated_tokens = output_ids[0][input_length:]
    predicted_answer = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    results.append({
        "question": q,
        "context": ctx,
        "ground_truth": true_answers,       
        "model_prediction": predicted_answer 
    })

# 4. Xuất toàn bộ kết quả ra file JSON
output_prediction_file = "/kaggle/working/test_predictions.json"
with open(output_prediction_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"\nĐã chạy xong! Kết quả lưu tại: {output_prediction_file}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Generating test split: 0 examples [00:00, ? examples/s]

Đang tiến hành Inference trên tập Test...


100%|██████████| 23/23 [00:08<00:00,  2.69it/s]


Đã chạy xong! Kết quả lưu tại: /kaggle/working/test_predictions.json
